#### Série: DuckDB Masterclass
**Ordem sugerida de leitura:** 5 de 14

# DuckDB Masterclass — Capítulo 05
## Window Functions, `QUALIFY`, `PIVOT` e `UNPIVOT`
**Formação:** Engenharia Analítica & Operations Analytics  
**Laboratório:** Jupyter + Python + DuckDB  
**Versão revisada para aprendizagem progressiva**

> **Objetivo:** sair da compreensão sintática para interpretar resultados, lidar com empates, controlar granularidade e validar decisões analíticas.

### Roteiro
1. Preparação e verificações
2. `ROW_NUMBER()` e `QUALIFY` — melhor/pior SLA por ERB
3. `PIVOT` e `UNPIVOT` — formato longo ↔ largo
4. Ranking com empates — `ROW_NUMBER`, `RANK`, `DENSE_RANK`
5. Acumulados — `ROWS` × `RANGE`
6. Participações percentuais — `Share of Total`
7. Laboratório integrado e desafio de negócios
8. Autoavaliação e referências oficiais

**Regra de execução:** rode as células de cima para baixo. Uma conexão `con` é criada apenas uma vez neste notebook. Dados fictícios, determinísticos, sem necessidade de internet para os labs.

> **Trilha de mentoria:** execute o capítulo em ordem, registre hipóteses e valide resultados. Para uma aula completa, consulte a seção de atividades de reflexão ao fim.

**Nota:** este capítulo já havia sido revisado; a versão foi preservada e apenas complementada.

## 05.0 — Preparação do ambiente
Se necessário, instale `duckdb` no seu ambiente Jupyter com `%pip install duckdb pandas` em uma célula separada. **Não** incluímos instalações automáticas no meio das análises; assim o notebook continua previsível e funciona em ambientes já preparados.

In [ ]:
import duckdb
import pandas as pd
from IPython.display import display

con = duckdb.connect(":memory:")
print("DuckDB:", duckdb.__version__)
print("Conexão criada: banco em memória da sessão atual")

## 05.2 — Window Functions e `QUALIFY`
**Pergunta de negócio:** qual foi a melhor observação de disponibilidade por ERB? E a pior?

`PARTITION BY erb` forma grupos lógicos; `ORDER BY` define a classificação **dentro** de cada ERB; `ROW_NUMBER` numera as linhas. `QUALIFY` filtra o resultado da janela sem exigir uma CTE extra.

**Cuidado de engenharia:** `QUALIFY` melhora a expressividade, mas **não garante uma única varredura física** nem velocidade superior a uma CTE. Uma ordenação secundária torna o resultado determinístico em caso de empates.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE sla_erbs AS
SELECT * FROM (VALUES
 ('ERB_1', DATE '2026-03-01', 98.20),
 ('ERB_1', DATE '2026-03-02', 99.50),
 ('ERB_1', DATE '2026-03-03', 99.50),
 ('ERB_2', DATE '2026-03-01', 97.10),
 ('ERB_2', DATE '2026-03-02', 99.10),
 ('ERB_2', DATE '2026-03-03', 96.80),
 ('ERB_3', DATE '2026-03-01', 96.40),
 ('ERB_3', DATE '2026-03-02', 98.00),
 ('ERB_3', DATE '2026-03-03', 97.50),
 ('ERB_4', DATE '2026-03-01', 99.20),
 ('ERB_4', DATE '2026-03-02', 98.60),
 ('ERB_4', DATE '2026-03-03', 98.90)
) AS t(erb, dt, disponibilidade_pct)
""")
display(con.execute("SELECT * FROM sla_erbs ORDER BY erb, dt").df())

In [ ]:
melhor_sla = con.execute("""
SELECT erb, dt, disponibilidade_pct,
       ROW_NUMBER() OVER (
           PARTITION BY erb
           ORDER BY disponibilidade_pct DESC, dt DESC
       ) AS rn
FROM sla_erbs
QUALIFY rn = 1
ORDER BY erb
""").df()
display(melhor_sla)
assert len(melhor_sla) == 4
assert melhor_sla.loc[melhor_sla.erb == 'ERB_1', 'dt'].iloc[0].day == 3
print('✓ Uma linha por ERB; empate em ERB_1 resolvido pela data mais recente.')

**Experimente:** altere `DESC` para `ASC` em `disponibilidade_pct`, mantendo `dt DESC`. O resultado passa a apontar a pior disponibilidade. Lembre-se: `ROW_NUMBER` entrega **uma** linha por partição; `RANK` pode preservar vários empates.

### Pergunta de fixação
Por que `WHERE rn = 1` não substitui `QUALIFY rn = 1` na mesma consulta? Porque `WHERE` atua antes do cálculo das funções de janela; `rn` resulta do janelamento.

## 05.3 — Remodelagem com `PIVOT` e `UNPIVOT`
- `PIVOT`: categorias que estavam em **linhas** passam a ser **colunas**, com eventual agregação.
- `UNPIVOT`: colunas passam a **linhas**, sem reconstruir automaticamente transações que foram agregadas.
- **PIVOT dinâmico:** se você omitir a lista de valores no `ON`, o DuckDB pode criar colunas para novas categorias. Ótimo para exploração, mas atenção ao contrato de esquema em BI/produção.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE chamados AS
SELECT * FROM (VALUES
 ('Norte', 'Jan', 120), ('Norte', 'Fev', 95),
 ('Sul', 'Jan', 80),   ('Sul', 'Fev', 110)
) AS t(regiao, mes, quantidade)
""")
res_pivot = con.execute("""
PIVOT chamados
ON mes
USING SUM(quantidade)
GROUP BY regiao
""").df()
display(res_pivot)
assert set(res_pivot.columns) == {'regiao', 'Jan', 'Fev'}

In [ ]:
res_unpivot = con.execute("""
UNPIVOT (
  PIVOT chamados ON mes USING SUM(quantidade) GROUP BY regiao
)
ON Jan, Fev
INTO NAME mes VALUE quantidade
""").df()
display(res_unpivot)
assert len(res_unpivot) == 4
print('✓ Matriz convertida em formato longo.')

### Experimento: PIVOT dinâmico e valores ausentes
Vamos adicionar março **somente para o Norte**. A nova coluna surge automaticamente, mas a região Sul terá `NULL` em março. `NULL` não significa necessariamente zero chamados; pode significar simplesmente ausência de registro.

In [ ]:
con.execute("INSERT INTO chamados VALUES ('Norte', 'Mar', 105)")
res_pivot_dinamico = con.execute("""
PIVOT chamados ON mes USING SUM(quantidade) GROUP BY regiao
""").df()
display(res_pivot_dinamico)
assert 'Mar' in res_pivot_dinamico.columns
assert pd.isna(res_pivot_dinamico.loc[res_pivot_dinamico.regiao == 'Sul', 'Mar']).all()
print('✓ Nova coluna Mar; ausência de Sul/Mar representada por NULL.')

**Observação:** `ON Jan, Fev` no UNPIVOT anterior é **estático** — não incluirá `Mar` automaticamente. Para contratos de saída estáveis, frequentemente é preferível listar categorias. Para seleção dinâmica de colunas, consulte `COLUMNS` na documentação oficial do `UNPIVOT`.

### Controle de granularidade
Se duas linhas tiverem a mesma região e mês, `SUM` as reúne em uma célula. O `UNPIVOT` dessa matriz **não recupera** as linhas de origem individualmente.

## 05.4.1 — Rankings e empates reais
- `ROW_NUMBER()`: sequência única (desempate explícito recomendado).
- `RANK()`: empate compartilha posição e a posição seguinte apresenta salto.
- `DENSE_RANK()`: empate compartilha posição **sem salto**.

O exemplo abaixo foi desenhado intencionalmente com empate, ao contrário de massas aleatórias que podem não produzi-lo.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE desempenho_filiais AS
SELECT * FROM (VALUES
 ('Campo Grande', 120), ('Dourados', 100), ('Três Lagoas', 100),
 ('Corumbá', 80), ('Ponta Porã', 60)
) AS t(filial, chamados_concluidos)
""")
res_rank = con.execute("""
SELECT filial, chamados_concluidos,
 ROW_NUMBER() OVER (ORDER BY chamados_concluidos DESC, filial ASC) AS posicao_unica,
 RANK() OVER (ORDER BY chamados_concluidos DESC) AS ranking,
 DENSE_RANK() OVER (ORDER BY chamados_concluidos DESC) AS ranking_denso
FROM desempenho_filiais
ORDER BY chamados_concluidos DESC, filial ASC
""").df()
display(res_rank)
assert list(res_rank.ranking) == [1,2,2,4,5]
assert list(res_rank.ranking_denso) == [1,2,2,3,4]
print('✓ Empate demonstrado e validado.')

**Cuidado:** a coluna `posicao_unica` utiliza um desempate por nome da filial. Já `RANK` e `DENSE_RANK` deliberadamente consideram **somente** o volume de chamados na ordenação, para manter o empate.

**Pergunta:** para identificar todas as filiais nos **três maiores níveis distintos** de produtividade, qual função é apropriada? `DENSE_RANK() ... QUALIFY ranking_denso <= 3`.

## 05.4.2 — Acumulados: `ROWS` versus `RANGE`
`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` acumula **linha a linha** na ordem definida. `RANGE` considera a faixa de valores e inclui pares (*peers*) com o mesmo valor de ordenação.

No exemplo a seguir há duas observações na mesma data. Com `ROWS`, incluímos o `id` como desempate para definir a sequência de maneira reproduzível; com `RANGE`, o ordenamento considera apenas `dt`.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE chamados_diarios AS
SELECT * FROM (VALUES
 (1, DATE '2026-10-01', 20),
 (2, DATE '2026-10-02', 35),
 (3, DATE '2026-10-02', 15)
) AS t(id, dt, quantidade)
""")
res_acumulados = con.execute("""
SELECT id, dt, quantidade,
 SUM(quantidade) OVER (
     ORDER BY dt, id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
 ) AS acumulado_rows,
 SUM(quantidade) OVER (
     ORDER BY dt RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
 ) AS acumulado_range
FROM chamados_diarios
ORDER BY dt, id
""").df()
display(res_acumulados)
assert list(res_acumulados.acumulado_rows) == [20,55,70]
assert list(res_acumulados.acumulado_range) == [20,70,70]
print('✓ ROWS e RANGE produzem resultados diferentes com datas empatadas.')

**Raciocínio aplicado:** acumulado por data é diferente de acumulado por valor decrescente. `ORDER BY data` mede evolução temporal; `ORDER BY receita DESC` mede concentração de receita, próxima à análise de Pareto. Sempre nomeie seu indicador de acordo com a interpretação de negócio.

## 05.4.3 — Participação percentual (Share of Total)
A participação é `valor / total_do_grupo × 100`.

`SUM(receita) OVER (PARTITION BY filial)` cria o denominador dentro de cada filial **sem eliminar** as linhas. `NULLIF(total, 0)` evita denominador zero, retornando `NULL` para participação indefinida.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE vendas_filiais AS
SELECT * FROM (VALUES
 ('Centro-Oeste', 'Produto_1', 1000.00),
 ('Centro-Oeste', 'Produto_2', 500.00),
 ('Centro-Oeste', 'Produto_3', 500.00),
 ('Norte', 'Produto_1', 900.00),
 ('Norte', 'Produto_2', 600.00),
 ('Nordeste', 'Produto_1', 700.00),
 ('Nordeste', 'Produto_2', 300.00)
) AS t(filial, produto, receita)
""")
res_integrado = con.execute("""
SELECT filial, produto, receita,
 ROW_NUMBER() OVER (PARTITION BY filial ORDER BY receita DESC, produto) AS rn_receita,
 RANK() OVER (PARTITION BY filial ORDER BY receita DESC) AS rank_receita,
 DENSE_RANK() OVER (PARTITION BY filial ORDER BY receita DESC) AS dense_rn_receita,
 SUM(receita) OVER (
  PARTITION BY filial ORDER BY receita DESC, produto
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
 ) AS receita_acumulada,
 ROUND(
  100.0 * receita / NULLIF(SUM(receita) OVER (PARTITION BY filial),0), 2
 ) AS share_filial_pct,
 ROUND(
  100.0 * SUM(receita) OVER (
    PARTITION BY filial ORDER BY receita DESC, produto
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) / NULLIF(SUM(receita) OVER (PARTITION BY filial),0),2
 ) AS share_acumulado_pct
FROM vendas_filiais
ORDER BY filial, receita DESC, produto
""").df()
display(res_integrado)
assert len(res_integrado) == 7
for filial, grupo in res_integrado.groupby('filial'):
    assert abs(grupo.share_filial_pct.sum()-100) < 0.03, filial
print('✓ Rankings, acumulado, share individual e share acumulado validados por filial.')

### Leitura executiva do laboratório integrado
- `rn_receita`: posição **única** do registro dentro da filial.
- `rank_receita` e `dense_rn_receita`: tornam visíveis os empates.
- `receita_acumulada`: concentração ordenada da maior para a menor receita; **não** é evolução temporal.
- `share_filial_pct`: contribuição de cada registro para a filial.
- `share_acumulado_pct`: quanto as primeiras linhas somam em percentual; útil em uma análise de concentração.

**Atenção à granularidade:** aqui cada linha é uma observação por produto/filial. Em dados transacionais, um produto pode aparecer várias vezes. Para uma análise de Pareto **por produto**, agregue primeiramente por filial e produto.

## 05.5 — Desafio de negócio: matriz de SLA por ERB
**Problema:** a gestão quer uma matriz com linhas por dia e colunas para ERB_1 a ERB_4, contendo a **média de disponibilidade** registrada naquele dia.

**Antes de executar a solução, tente escrever o SQL sozinho.** Pense em:
1. Tabela-fonte: `sla_erbs` (criada no item 5.2 e ainda presente na mesma conexão).
2. Categorias de colunas: ERB_1, ERB_2, ERB_3, ERB_4.
3. Função de agregação: `AVG(disponibilidade_pct)`.
4. Granularidade das linhas: `dt`.

In [ ]:
# Solução de referência: execute após tentar resolver
matriz_sla = con.execute("""
PIVOT sla_erbs
ON erb IN ('ERB_1', 'ERB_2', 'ERB_3', 'ERB_4')
USING ROUND(AVG(disponibilidade_pct), 2)
GROUP BY dt
ORDER BY dt
""").df()
display(matriz_sla)
assert set(matriz_sla.columns) == {'dt','ERB_1','ERB_2','ERB_3','ERB_4'}
assert len(matriz_sla) == 3
print('✓ Matriz de SLA coerente com os identificadores da tabela de origem.')

### Desafio extra — além da sintaxe
- Qual ERB tem a **menor média** no período? Antes de fazer `PIVOT`, você consegue responder com `GROUP BY erb`?
- Em uma tabela operacional real, ausência de registro (`NULL`) pode ser interpretada como SLA 0%? **Não sem uma regra de negócio explícita.**
- Qual é o risco de usar um `PIVOT` dinâmico em uma tabela consumida por dashboards com nomes de coluna fixos?

**Pergunta de qualidade:** como garantir desempate consistente se duas observações da mesma ERB tiverem igual disponibilidade e igual data? Adicione um identificador único da observação e use-o como critério final de ordenação.

## 05.6 — Verificação de domínio (responda antes de abrir a solução)
1. Em `QUALIFY rn = 1`, o que muda se removemos `PARTITION BY erb`?
2. Para empates `100, 100, 80`, qual resultado de `RANK` e `DENSE_RANK`?
3. Quando `RANGE` pode diferir de `ROWS`?
4. Por que `OVER (PARTITION BY filial)` não reduz linhas como um `GROUP BY filial`?
5. Por que `UNPIVOT` não recupera transações individuais após um `PIVOT SUM`?
6. Qual a diferença entre participação individual e participação acumulada?

<details>
<summary><strong>Gabarito comentado — consulte somente após responder</strong></summary>

1. Haverá apenas uma partição global; retorna uma linha no total.
2. `RANK = 1, 1, 3`; `DENSE_RANK = 1, 1, 2`.
3. Quando há pares com valor de ordenação igual: `RANGE` pode somá-los conjuntamente.
4. Funções de janela preservam a granularidade das linhas de entrada.
5. A agregação descarta informação sobre o detalhamento original.
6. Individual = valor da linha dividido pelo total do grupo; acumulada = soma progressiva dividida pelo total do grupo.

</details>

## 05.7 — Conclusão e referências
### Competências consolidadas
- Diferenciar o **resultado lógico** de `QUALIFY` do plano físico da consulta.
- Escolher a função de ranking conforme a política de empates.
- Entender o comportamento de `ROWS` e `RANGE`.
- Calcular participação individual e acumulada, respeitando a granularidade.
- Usar `PIVOT`/`UNPIVOT`, incluindo o risco de esquemas dinâmicos.
- Validar resultados com `assert` e dados reproduzíveis.

### Documentação oficial
- [DuckDB — QUALIFY](https://duckdb.org/docs/stable/sql/query_syntax/qualify)
- [DuckDB — Window Functions](https://duckdb.org/docs/stable/sql/functions/window_functions.html)
- [DuckDB — PIVOT](https://duckdb.org/docs/stable/sql/statements/pivot)
- [DuckDB — UNPIVOT](https://duckdb.org/docs/stable/sql/statements/unpivot)
- [DuckDB — EXPLAIN](https://duckdb.org/docs/stable/guides/meta/explain.html)

**Próximo passo sugerido:** interpretar planos de execução com `EXPLAIN` e comparar alternativas equivalentes, sem pressupor que SQL mais curto implica execução mais rápida.

## 🔎 Avaliação de especialista — reflexão antes da próxima aula

1. Qual decisão de negócio o laboratório permite tomar?
2. Que hipótese a consulta assume sobre os dados?
3. Que teste automatizado detectaria uma violação dessa hipótese?
4. Como explicar o resultado sem consultar o código?

Anote as respostas em uma nova célula Markdown.